In [5]:
!pip install langchain langchain-community chromadb pypdf sentence-transformers torch transformers accelerate rank_bm25

import os
from typing import List, Dict, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import re
import uuid
from rank_bm25 import BM25Okapi


2025-11-15 12:39:37.222474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763210377.414491      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763210377.473698      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [6]:
!pip install -U sentence-transformers

In [26]:
import os
import re
from typing import List, Optional
import torch
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document
from rank_bm25 import BM25Okapi


class LegalSearchAgent:
    # =======================================================================
    #                         INITIALIZATION
    # =======================================================================
    def __init__(self, pdf_folder: str, embeddings: HuggingFaceEmbeddings, db_path: str = "chroma_db", test_mode: bool = True):
        self.pdf_folder = pdf_folder
        self.db_path = db_path
        self.test_mode = test_mode
        self.embeddings = embeddings

        self.vectorstore: Optional[Chroma] = None
        self.bm25: Optional[BM25Okapi] = None
        self.bm25_docs: List[Document] = []
        self.bm25_corpus: List[List[str]] = []

    # =======================================================================
    #                         BUILD VECTOR DATABASE
    # =======================================================================
    def build_vectordb(self):
        all_docs: List[Document] = []

        pdf_files = [f for f in os.listdir(self.pdf_folder) if f.lower().endswith(".pdf") and len(f) > 5]
        if self.test_mode:
            pdf_files = pdf_files[:5]
            print(f"⚠️ TEST MODE: Using only first {len(pdf_files)} PDFs")
        else:
            print(f"Found {len(pdf_files)} PDFs to process.")

        # -------------------------------
        # LOAD PDFs AND ATTACH METADATA
        # -------------------------------
        for idx, pdf in enumerate(pdf_files, 1):
            try:
                print(f"📄 Loading [{idx}/{len(pdf_files)}]: {pdf}")
                loader = PyPDFLoader(os.path.join(self.pdf_folder, pdf))
                pages = loader.load()

                base = pdf.replace(".pdf", "")
                parts = re.split(r"[_\-]", base)
                case_type = "_".join(parts[:-1]) if len(parts) >= 2 else base
                case_year = parts[-1] if parts[-1].isdigit() else "unknown"

                for page in pages:
                    page.metadata["source_file"] = pdf
                    page.metadata["case_number"] = base
                    page.metadata["case_year"] = case_year
                    page.metadata["case_type"] = case_type

                all_docs.extend(pages)

            except Exception as e:
                print(f"❌ Error loading {pdf}: {e}")

        print(f"Loaded {len(all_docs)} pages. Splitting...")

        # -------------------------------
        # SPLIT TEXT AND INJECT METADATA
        # -------------------------------
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ".", " ", ""]
        )

        chunks: List[Document] = []
        for page in all_docs:
            split_chunks = splitter.split_documents([page])
            for ch in split_chunks:
                # Copy metadata from original page
                ch.metadata = page.metadata.copy()
                case_type = page.metadata.get("case_type", "")
                case_year = page.metadata.get("case_year", "")
                case_number = page.metadata.get("case_number", "")

                # Inject metadata into text for semantic search
                ch.page_content = f"[Case Type: {case_type}] [Year: {case_year}] [Case Number: {case_number}]\n\n{ch.page_content}"
                chunks.append(ch)

        print(f"✂️ Split into {len(chunks)} chunks.")
        print("🔮 Generating embeddings and building vector DB...")

        # -------------------------------
        # BUILD CHROMA VECTORSTORE
        # -------------------------------
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.db_path
        )
        print(f"✅ Chroma DB saved at {self.db_path}")

        # -------------------------------
        # BUILD BM25 INDEX
        # -------------------------------
        self.bm25_docs = chunks
        self.bm25_corpus = [ch.page_content.split() for ch in chunks]
        self.bm25 = BM25Okapi(self.bm25_corpus)
        print("📌 BM25 index ready.\n")

    # =======================================================================
    #                         CASE NUMBER DETECTION
    # =======================================================================
    def _detect_case_number(self, query: str) -> Optional[str]:
        pattern = r"(CPLA|C\.A\.|Cr\.A|C\.P\.|HCA|RFA)[\s\-]*\d+[\s/]*(?:of\s*)?\d{4}"
        match = re.search(pattern, query, re.IGNORECASE)
        if match:
            case = match.group(0)
            case = case.replace(" ", "_").replace("of_", "_").replace("/", "_")
            case = re.sub(r"__+", "_", case)
            return case
        return None

    # =======================================================================
    #                         HYBRID RETRIEVER
    # =======================================================================
    def _retrieve_documents(self, query: str, k: int = 10) -> List[Document]:
        if self.vectorstore is None or self.bm25 is None:
            print("⚠️ Vectorstore or BM25 not built yet.")
            return []

        # Semantic search
        semantic_results = self.vectorstore.similarity_search(query, k=k)

        # BM25 keyword search
        tokens = query.split()
        bm25_scores = self.bm25.get_scores(tokens)
        bm25_top_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:k]
        bm25_results = [self.bm25_docs[i] for i in bm25_top_idx]

        # Merge results (remove duplicates by case_number)
        combined = {}
        for doc in semantic_results + bm25_results:
            key = doc.metadata.get("case_number", doc.metadata.get("source_file", id(doc)))
            if key not in combined:
                combined[key] = doc

        return list(combined.values())[:k]

    # =======================================================================
    #                         MAIN SEARCH
    # =======================================================================
    def search(self, query: str, k: int = 10) -> List[Document]:
        print(f"\n🔍 QUERY: {query}")

        case_num = self._detect_case_number(query)
        if case_num:
            print(f"📋 Detected case number: {case_num}")
            parts = case_num.split("_")
            if len(parts) >= 2:
                case_number_only = parts[-2]
                case_year = parts[-1]

                # Get more results to filter from
                results = self._retrieve_documents(query, k=k*3)
                # Filter to exact case match (by number and year)
                exact_match = [
                    r for r in results
                    if case_number_only in r.metadata.get('case_number', '') and case_year == r.metadata.get('case_year', '')
                ]
                if exact_match:
                    results = exact_match[:k]
                    print(f"✅ Found exact case match")
                else:
                    print(f"⚠️ No exact case found, returning semantic matches")
                    results = results[:k]
            else:
                results = self._retrieve_documents(query, k=k)
        else:
            results = self._retrieve_documents(query, k=k)

        print("\n📄 Retrieved PDFs:")
        for r in results:
            print(" -", r.metadata.get("source_file", "unknown"))

        return results


In [28]:
# -------------------------------
# 1️⃣ IMPORTS
# -------------------------------
import torch
from langchain.embeddings import HuggingFaceEmbeddings

# -------------------------------
# 2️⃣ SETUP EMBEDDINGS
# -------------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-distilroberta-base-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# -------------------------------
# 3️⃣ CREATE LEGAL SEARCH AGENT
# -------------------------------
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",   # path to your folder containing PDFs
    db_path="chroma_db", # folder to store vector DB
    embeddings=embeddings,
    test_mode=False       # set True to only process first 5 PDFs
)

agent.vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

#agent.build_vectordb()

# -------------------------------
# 4️⃣ RUN SEARCH
# -------------------------------
query1 = "What was CPLA 210 of 2024 about?"
results = agent.search(query1, k=15)

print("\n--- RESULTS ---")
for r in results:
    print(f"{r.metadata['case_number']} | {r.metadata['source_file']} | {r.metadata['case_year']}")



🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024
⚠️ Vectorstore or BM25 not built yet.
⚠️ No exact case found, returning semantic matches

📄 Retrieved PDFs:

--- RESULTS ---


In [30]:
# ========================================
# 1️⃣ SETUP EMBEDDINGS
# ========================================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-distilroberta-base-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# ========================================
# 2️⃣ CREATE AGENT (don't build, just initialize)
# ========================================
agent = LegalSearchAgent(
    pdf_folder="/kaggle/input/fyp-data/supreme_court_judgments",
    db_path="chroma_db",
    embeddings=embeddings,
    test_mode=False
)

# ========================================
# 3️⃣ LOAD EXISTING VECTORSTORE
# ========================================
print("Loading existing vector store...")
agent.vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)
print(f"✅ Loaded {agent.vectorstore._collection.count()} chunks from vector DB")

# ========================================
# 4️⃣ REBUILD BM25 FROM EXISTING CHUNKS
# ========================================
print("Rebuilding BM25 index from vectorstore...")
vectorstore_data = agent.vectorstore.get()

docs = []
for i, content in enumerate(vectorstore_data['documents']):
    metadata = vectorstore_data['metadatas'][i]
    doc = Document(page_content=content, metadata=metadata)
    docs.append(doc)

agent.bm25_docs = docs
agent.bm25_corpus = [doc.page_content.split() for doc in docs]
agent.bm25 = BM25Okapi(agent.bm25_corpus)
print(f"✅ BM25 ready with {len(agent.bm25_docs)} documents")

# ========================================
# 5️⃣ NOW TEST SEARCH
# ========================================
query1 = "What was CPLA 210 of 2024 about?"
results = agent.search(query1, k=15)

print("\n--- RESULTS ---")
for r in results:
    print(f"{r.metadata['case_number']} | {r.metadata['source_file']} | {r.metadata['case_year']}")

Loading existing vector store...
✅ Loaded 87580 chunks from vector DB
Rebuilding BM25 index from vectorstore...
✅ BM25 ready with 87580 documents

🔍 QUERY: What was CPLA 210 of 2024 about?
📋 Detected case number: CPLA_210_2024
⚠️ No exact case found, returning semantic matches

📄 Retrieved PDFs:
 - Crl.P.L.A.80-P_2024.pdf
 - C.P.L.A.2226-L_2021.pdf
 - C.P.L.A.385-L_2021.pdf
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.1369-L_2022.pdf
 - C.P.4_2021.pdf
 - C.P.L.A.5438_2021.pdf
 - C.P.L.A.3644_2020.pdf
 - Crl.P.L.A.457-L_2021.pdf
 - Crl.P.L.A.1072_2021.pdf
 - Crl.P.L.A.112_2020.pdf
 - C.M.A.1609-L_2021.pdf
 - C.P.L.A.4806_2019.pdf
 - C.A.538_2022.pdf
 - C.P.L.A.81-P_2019.pdf

--- RESULTS ---
Crl.P.L.A.80-P_2024 | Crl.P.L.A.80-P_2024.pdf | 2024
C.P.L.A.2226-L_2021 | C.P.L.A.2226-L_2021.pdf | 2021
C.P.L.A.385-L_2021 | C.P.L.A.385-L_2021.pdf | 2021
C.P.L.A.279-Q_2020 | C.P.L.A.279-Q_2020.pdf | 2020
C.P.L.A.1369-L_2022 | C.P.L.A.1369-L_2022.pdf | 2022
C.P.4_2021 | C.P.4_2021.pdf | 2021
C.P.L.A.5438_

In [31]:
query1 = "What was C.P.L.A 210 of 2024 about?"
results = agent.search(query1, k=15)

print("\n--- RESULTS ---")
for r in results:
    print(f"{r.metadata['case_number']} | {r.metadata['source_file']} | {r.metadata['case_year']}")


🔍 QUERY: What was C.P.L.A 210 of 2024 about?

📄 Retrieved PDFs:
 - Crl.P.L.A.80-P_2024.pdf
 - C.P.L.A.279-Q_2020.pdf
 - C.P.L.A.385-L_2021.pdf
 - C.P.L.A.2226-L_2021.pdf
 - C.P.L.A.1369-L_2022.pdf
 - Crl.P.L.A.1072_2021.pdf
 - C.P.L.A.184_2024.pdf
 - Crl.P.L.A.457-L_2021.pdf
 - C.R.P.870_2023.pdf
 - C.P.L.A.3531_2021.pdf
 - C.P.L.A.6-L_2023.pdf

--- RESULTS ---
Crl.P.L.A.80-P_2024 | Crl.P.L.A.80-P_2024.pdf | 2024
C.P.L.A.279-Q_2020 | C.P.L.A.279-Q_2020.pdf | 2020
C.P.L.A.385-L_2021 | C.P.L.A.385-L_2021.pdf | 2021
C.P.L.A.2226-L_2021 | C.P.L.A.2226-L_2021.pdf | 2021
C.P.L.A.1369-L_2022 | C.P.L.A.1369-L_2022.pdf | 2022
Crl.P.L.A.1072_2021 | Crl.P.L.A.1072_2021.pdf | 2021
C.P.L.A.184_2024 | C.P.L.A.184_2024.pdf | 2024
Crl.P.L.A.457-L_2021 | Crl.P.L.A.457-L_2021.pdf | 2021
C.R.P.870_2023 | C.R.P.870_2023.pdf | 2023
C.P.L.A.3531_2021 | C.P.L.A.3531_2021.pdf | 2021
C.P.L.A.6-L_2023 | C.P.L.A.6-L_2023.pdf | 2023


In [32]:
results = agent._retrieve_documents("What was CPLA 210 of 2024 about?", k=30)
case_num = agent._detect_case_number("What was CPLA 210 of 2024 about?")

print(f"Detected case_num: {case_num}")
print(f"Looking for: number=210, year=2024")

# Extract parts
parts = case_num.split("_")
case_number_only = parts[-2] if len(parts) >= 2 else None
case_year = parts[-1] if len(parts) >= 1 else None

print(f"Extracted: case_number={case_number_only}, year={case_year}")

# Filter manually
for r in results[:5]:
    has_number = case_number_only in r.metadata['case_number']
    has_year = case_year == r.metadata['case_year']
    print(f"{r.metadata['case_number']} | year={r.metadata['case_year']} | matches={has_number and has_year}")

Detected case_num: CPLA_210_2024
Looking for: number=210, year=2024
Extracted: case_number=210, year=2024
Crl.P.L.A.80-P_2024 | year=2024 | matches=False
C.P.L.A.2226-L_2021 | year=2021 | matches=False
C.P.L.A.385-L_2021 | year=2021 | matches=False
C.P.L.A.279-Q_2020 | year=2020 | matches=False
C.P.L.A.1369-L_2022 | year=2022 | matches=False


In [35]:
# Test embedding quality
models = [
    "sentence-transformers/paraphrase-distilroberta-base-v2",
    "BAAI/bge-large-en-v1.5",
]

for model_name in models:
    emb = HuggingFaceEmbeddings(model_name=model_name)
    
    # Embed query and target
    q = emb.embed_query("[Case Type: C.P.L.A.210] [Year: 2024] [Case Number: C.P.L.A.210_2024]")
    doc = emb.embed_query("What was CPLA 210 of 2024 about?")
    
    # Cosine similarity
    import numpy as np
    sim = np.dot(q, doc) / (np.linalg.norm(q) * np.linalg.norm(doc))
    print(f"{model_name}: {sim:.4f}")

sentence-transformers/paraphrase-distilroberta-base-v2: 0.5284
BAAI/bge-large-en-v1.5: 0.7540


In [36]:
import numpy as np
from langchain.embeddings import HuggingFaceEmbeddings

# Test queries with expected case metadata
test_cases = [
    {
        "query": "What was CPLA 210 of 2024 about?",
        "metadata": "[Case Type: C.P.L.A.210] [Year: 2024] [Case Number: C.P.L.A.210_2024]"
    },
    {
        "query": "income tax ordinance section 151",
        "metadata": "[Case Type: C.P.L.A.3578] [Year: 2024] [Case Number: C.P.L.A.3578_2024]\n\nIncome Tax Ordinance Section 122(5A)"
    },
    {
        "query": "writ petition not maintainable",
        "metadata": "[Case Type: C.P.L.A] [Year: 2024]\n\nwrit petition was held to be not maintainable"
    },
    {
        "query": "regularization notification 2011",
        "metadata": "[Case Type: C.P.L.A] [Year: various]\n\nregularization notification of 2011"
    },
    {
        "query": "Civil Petition for Leave to Appeal 4424 of 2021",
        "metadata": "[Case Type: C.P.L.A.4424] [Year: 2021] [Case Number: C.P.L.A.4424_2021]"
    }
]

# Models to test
models = [
    "sentence-transformers/paraphrase-distilroberta-base-v2",
    "nlpaueb/legal-bert-base-uncased",
    "BAAI/bge-large-en-v1.5",
    "sentence-transformers/legal-distilroberta-base",
    "sentence-transformers/all-mpnet-base-v2",
]

print("=" * 80)
print("EMBEDDING MODEL COMPARISON")
print("=" * 80)

results_summary = {}

for model_name in models:
    print(f"\n📊 Testing: {model_name}")
    print("-" * 80)
    
    try:
        emb = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cuda'})
        
        similarities = []
        
        for i, test_case in enumerate(test_cases, 1):
            query = test_case["query"]
            metadata = test_case["metadata"]
            
            try:
                q_emb = emb.embed_query(query)
                doc_emb = emb.embed_query(metadata)
                
                # Cosine similarity
                sim = np.dot(q_emb, doc_emb) / (np.linalg.norm(q_emb) * np.linalg.norm(doc_emb))
                similarities.append(sim)
                
                print(f"  {i}. {query[:50]:50s} → {sim:.4f}")
            except Exception as e:
                print(f"  {i}. {query[:50]:50s} → ERROR: {e}")
                similarities.append(0)
        
        avg_sim = np.mean(similarities)
        results_summary[model_name] = avg_sim
        print(f"  📈 AVERAGE: {avg_sim:.4f}")
        
    except Exception as e:
        print(f"  ❌ FAILED TO LOAD: {e}")
        results_summary[model_name] = 0

print("\n" + "=" * 80)
print("SUMMARY RANKING")
print("=" * 80)

sorted_models = sorted(results_summary.items(), key=lambda x: x[1], reverse=True)
for i, (model, avg_sim) in enumerate(sorted_models, 1):
    print(f"{i}. {model:60s} → {avg_sim:.4f}")

print("\n" + "=" * 80)
print(f"🏆 BEST MODEL: {sorted_models[0][0]}")
print("=" * 80)

EMBEDDING MODEL COMPARISON

📊 Testing: sentence-transformers/paraphrase-distilroberta-base-v2
--------------------------------------------------------------------------------


  1. What was CPLA 210 of 2024 about?                   → 0.5284
  2. income tax ordinance section 151                   → 0.4712
  3. writ petition not maintainable                     → 0.6308
  4. regularization notification 2011                   → 0.6398
  5. Civil Petition for Leave to Appeal 4424 of 2021    → 0.4633
  📈 AVERAGE: 0.5467

📊 Testing: nlpaueb/legal-bert-base-uncased
--------------------------------------------------------------------------------


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  1. What was CPLA 210 of 2024 about?                   → 0.5095
  2. income tax ordinance section 151                   → 0.6629
  3. writ petition not maintainable                     → 0.7776
  4. regularization notification 2011                   → 0.6996
  5. Civil Petition for Leave to Appeal 4424 of 2021    → 0.4546
  📈 AVERAGE: 0.6209

SUMMARY RANKING
1. BAAI/bge-large-en-v1.5                                       → 0.7749
2. nlpaueb/legal-bert-base-uncased                              → 0.7391
3. sentence-transformers/all-mpnet-base-v2                      → 0.6209
4. sentence-transformers/paraphrase-distilroberta-base-v2       → 0.5467
5. sentence-transformers/legal-distilroberta-base               → 0.0000

🏆 BEST MODEL: BAAI/bge-large-en-v1.5


In [27]:
# Check if the case exists in your DB
from langchain.vectorstores import Chroma

vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

# Search directly in metadata
all_data = vectorstore.get()
cpla_210_cases = [m for m in all_data['metadatas'] if 'C.P.L.A.210_2024' in m.get('case_number', '')]
print(f"Found {len(cpla_210_cases)} chunks from C.P.L.A.210_2024")

/tmp/ipykernel_48/1621021105.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


Found 12 chunks from C.P.L.A.210_2024


In [12]:
PDF_FOLDER = "/kaggle/input/fyp-data/supreme_court_judgments"

print("Starting LOCAL Legal Search Agent\n")
print("No API keys, No Ollama - runs 100% locally!\n")


agent = LegalSearchAgent(
    pdf_folder=PDF_FOLDER,
    test_mode=False
)

agent.build_vectordb()

print("\n" + "="*80)
print("LEGAL SEARCH AGENT READY (LOCAL)")
print("="*80)


Starting LOCAL Legal Search Agent

No API keys, No Ollama - runs 100% locally!



TypeError: LegalSearchAgent.__init__() missing 1 required positional argument: 'embeddings'

In [ ]:
print("\nType your query (or 'quit' to exit)")
print("Example: 'Why did the court reject SIC's appeal?'\n")

while True:
    query = input("\n💬 Your query: ").strip()
    
    if query.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Goodbye!")
        break
    
    if not query:
        continue
    
    try:
        result = agent.search(query)
        
        print("\n📎 Relevant PDFs:")
        for pdf in result['relevant_pdfs']:
            print(f"   - {pdf}")
        print()
    except Exception as e:
        print(f"❌ Error: {e}")
        print("Please try again or check your setup.\n")


Type your query (or 'quit' to exit)
Example: 'Why did the court reject SIC's appeal?'




💬 Your query:  What was CPLA 210 of 2024 about?



🔍 QUERY: What was CPLA 210 of 2024 about?

📄 Retrieved:
❌ Error: 'source_file'
Please try again or check your setup.




💬 Your query:  quit


In [25]:
agent = LegalSearchAgent(
    pdf_folder=PDF_FOLDER,
    test_mode=False
)
agent.build_vectordb()


🤖 Loading local LLM (Phi-2)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Local LLM loaded successfully.
📚 Loading Sentence-BERT embeddings...
Sentence-BERT loaded.

📚 Loading existing Chroma DB...
Loaded 21895 chunks.
Rebuilding BM25 index...


❌ Error loading Crl.A.306-L_2012.pdf: Invalid Elementary Object starting with b')' @12714: b'-9.9888371(h)17.56n:)-S953.72298346.2883S1(s)20.33(&)-9.988837( )-203.471(e)-27.'


❌ Error loading C.P.L.A.2743_2017.pdf: Invalid Elementary Object starting with b'I' @22327: b'l)3123.3n)19( )98 0.IT\n/F2 12.0 Tf\n 0.0 0.0 rg\n0.9998 137(s)8( )-70(ne2(h)19(e)3'


📌 BM25 index rebuilt.

